In [1]:
import os
import time
import truststore

# Use macOS system trust store (includes corporate CA certs)
truststore.inject_into_ssl()

os.environ["LANGCHAIN_PROJECT"] = "demo-optimization"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [2]:
from langsmith import Client

client = Client()

In [3]:
# In the original notebook, an automation rule collects "correct" traces into a dataset.
# Since we don't have pre-existing project/dataset UUIDs, we'll create the dataset manually
# after running some queries and marking them as correct.

ds_name = "demo-optimization"
try:
    ds = client.read_dataset(dataset_name=ds_name)
    print(f"Dataset '{ds_name}' already exists.")
except Exception:
    ds = client.create_dataset(dataset_name=ds_name)
    print(f"Created dataset '{ds_name}'.")

Created dataset 'demo-optimization'.


In [4]:
import openai as openai_mod
from langsmith import wrappers, traceable

groq_client = wrappers.wrap_openai(openai_mod.Client(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    ))


In [5]:
import json

tools = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_movies",
            "description": "Retrieve a list of relevant movies and their metadata from a movie database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The query used to retrieve movies from the movie database, for example 'Christopher Nolan films'",
                    },
                },
                "required": ["query"],
            },
        },
    },
]

system_prompt = """
Don't make assumptions about what values to plug into functions. Ask for clarification if a user request is ambiguous.
Note that if the question does not require additional search and can be answered using the chat history, simply respond with the answer.
Don't make up content that's not supplied in chat history.
"""


@traceable
def generate_movie_search(messages):
    messages = [
        {"role": "system", "content": system_prompt},
    ] + messages
    time.sleep(3)
    result = groq_client.chat.completions.create(
        messages=messages, model="llama-3.1-8b-instant", tools=tools
    )
    return result.choices[0].message


def _convert_docs(results):
    return [
        {
            "page_content": r,
            "type": "Document",
        }
        for r in results
    ]


@traceable(run_type="retriever")
def retrieve_movies(query):
    # Mock retriever. In production, this would search an actual database
    if "family" in query.lower():
        return _convert_docs(["Lion King", "Finding Nemo", "The Incredibles"])
    elif "sci-fi" in query.lower() or "alien" in query.lower():
        return _convert_docs(["Blade Runner 2049", "Interstellar", "The Martian"])
    elif "nature" in query.lower():
        return _convert_docs(["Planet Earth II", "Blue Planet II", "Our Planet"])
    elif "christopher nolan" in query.lower():
        return _convert_docs(["Inception", "Dunkirk", "Interstellar"])
    elif "suspense" in query.lower() or "thriller" in query.lower():
        return _convert_docs(["Se7en", "Gone Girl", "Zodiac"])
    else:
        return _convert_docs(
            ["Crazy Rich Asians", "The Big Sick", "When Harry Met Sally"]
        )


@traceable
def execute_function_call(message):
    if message.tool_calls[0].function.name == "retrieve_movies":
        query = json.loads(message.tool_calls[0].function.arguments)["query"]
        results = retrieve_movies(query)
    else:
        results = (
            f"Error: function {message.tool_calls[0].function.name} does not exist"
        )
    return results


@traceable
def generate_answer(messages, context):
    messages = [
        {
            "role": "system",
            "content": f"Answer the user's question based only on the content below:\n\n{context}",
        },
    ] + messages
    time.sleep(3)
    result = groq_client.chat.completions.create(
        messages=messages, model="llama-3.1-8b-instant", temperature=0
    )
    return result.choices[0].message.content


@traceable
def rag_pipeline(messages):
    message = generate_movie_search(messages)
    if message.tool_calls is None:
        return message.content
    else:
        docs = execute_function_call(message)
        context = "\n".join([doc["page_content"] for doc in docs])
        return generate_answer(messages, context)

In [6]:
rag_pipeline([{"role": "user", "content": "what are some family movies to watch?"}])

"Here are a few family movie suggestions:\n\n1. Lion King - an animated classic about a young lion's journey to become king of the Pride Lands.\n2. Finding Nemo - a heartwarming story about a clownfish's quest to find his son in the vast ocean.\n3. The Incredibles - a superhero animated film about a family with superpowers trying to live a normal life.\n\nThese movies are all highly rated and suitable for a family movie night."

In [7]:
rag_pipeline([{"role": "user", "content": "what are some movies about aliens?"}])

'Based on the given information, I can suggest two movies that involve aliens:\n\n1. Blade Runner 2049 - This movie is a sequel to the original Blade Runner and involves replicants (androids that are nearly indistinguishable from humans) and a plot that explores the possibility of life beyond Earth. While not strictly about aliens, it does involve a search for a new home for humanity and encounters with a mysterious, possibly extraterrestrial, presence.\n\n2. Interstellar - This movie involves a team of astronauts who travel through a wormhole in search of a new home for humanity as Earth faces impending environmental disaster. They encounter strange occurrences and possibly alien life forms during their journey.\n\nNeither of these movies is strictly about aliens, but they both involve themes of space exploration and the possibility of extraterrestrial life.'

In [8]:
rag_pipeline([{"role": "user", "content": "Need a light-hearted movie"}])

'Out of the three options, I would recommend "When Harry Met Sally". It\'s a classic romantic comedy that\'s light-hearted, funny, and has a great cast, including Billy Crystal and Meg Ryan. It\'s a great choice if you\'re looking for a feel-good movie that will leave you smiling.'

In [10]:
# Collect the successful runs and add as dataset examples
# Wait for traces to flush to LangSmith
import time as _t
_t.sleep(10)

# Get top-level runs (rag_pipeline) that used tool calls
runs = list(client.list_runs(
    project_name="demo-optimization",
    filter='eq(name, "generate_movie_search")',
))

print(f"Found {len(runs)} generate_movie_search runs")

# Add runs with tool_calls as examples
added = 0
for run in runs:
    if run.inputs and run.outputs:
        try:
            client.create_example(
                inputs=run.inputs,
                outputs=run.outputs,
                dataset_id=ds.id,
            )
            added += 1
        except Exception as e:
            pass

examples = list(client.list_examples(dataset_name=ds_name))
print(f"Added {added} examples. Dataset now has {len(examples)} examples.")

Found 3 generate_movie_search runs
Added 3 examples. Dataset now has 3 examples.


In [11]:
if examples:
    print("Example inputs:", examples[0].inputs)
else:
    print("No examples yet")

Example inputs: {'messages': [{'role': 'user', 'content': 'what are some family movies to watch?'}]}


In [12]:
if examples:
    print("Example outputs:", examples[0].outputs)
else:
    print("No examples yet")

Example outputs: {'role': 'assistant', 'tool_calls': [{'id': '0y26qpnkz', 'type': 'function', 'function': {'name': 'retrieve_movies', 'arguments': '{"query":"family movies to watch"}'}}]}


In [17]:
few_shot_examples = []
for example in examples:
    # Add the user messages from the input
    if "messages" in example.inputs:
        few_shot_examples.extend(example.inputs["messages"])
    # Add the assistant output with tool calls
    if example.outputs and "tool_calls" in example.outputs:
        tc = example.outputs["tool_calls"]
        # Groq needs tool_calls in exact format with content=None
        assistant_msg = {
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {
                    "id": call["id"],
                    "type": "function",
                    "function": {
                        "name": call["function"]["name"],
                        "arguments": call["function"]["arguments"],
                    }
                }
                for call in tc
            ]
        }
        few_shot_examples.append(assistant_msg)
        # Add tool responses
        for call in tc:
            query = json.loads(call["function"]["arguments"]).get("query", "")
            # Get actual retrieval results for this query
            results = retrieve_movies(query)
            few_shot_examples.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "content": json.dumps([r["page_content"] for r in results]),
            })
    elif example.outputs and "role" in example.outputs:
        # Flat format
        output = example.outputs
        if output.get("tool_calls"):
            assistant_msg = {
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {
                        "id": call["id"],
                        "type": "function",
                        "function": {
                            "name": call["function"]["name"],
                            "arguments": call["function"]["arguments"],
                        }
                    }
                    for call in output["tool_calls"]
                ]
            }
            few_shot_examples.append(assistant_msg)
            for call in output["tool_calls"]:
                query = json.loads(call["function"]["arguments"]).get("query", "")
                results = retrieve_movies(query)
                few_shot_examples.append({
                    "role": "tool",
                    "tool_call_id": call["id"],
                    "content": json.dumps([r["page_content"] for r in results]),
                })

print(f"Built {len(few_shot_examples)} few-shot messages")
for m in few_shot_examples:
    print(f"  {m['role']}: {str(m.get('content', m.get('tool_calls', '')))[:80]}")

Built 9 few-shot messages
  user: what are some family movies to watch?
  assistant: None
  tool: ["Lion King", "Finding Nemo", "The Incredibles"]
  user: what are some movies about aliens?
  assistant: None
  tool: ["Blade Runner 2049", "Interstellar", "The Martian"]
  user: Need a light-hearted movie
  assistant: None
  tool: ["Crazy Rich Asians", "The Big Sick", "When Harry Met Sally"]


In [18]:
@traceable
def generate_movie_search(messages):
    messages = (
        [
            {"role": "system", "content": system_prompt},
        ]
        + few_shot_examples
        + messages
    )
    time.sleep(3)
    result = groq_client.chat.completions.create(
        messages=messages, model="llama-3.1-8b-instant", tools=tools
    )
    return result.choices[0].message

In [19]:
@traceable
def rag_pipeline(messages):
    message = generate_movie_search(messages)
    if message.tool_calls is None:
        return message.content
    else:
        docs = execute_function_call(message)
        context = "\n".join([doc["page_content"] for doc in docs])
        return generate_answer(messages, context)

In [20]:
result = rag_pipeline([{"role": "user", "content": "Suggest a suspenseful movie"}])
print(result)

I'd recommend "Se7en". It's a psychological thriller that follows two detectives as they hunt for a serial killer who uses the seven deadly sins as a motif for his murders. The movie is known for its dark and suspenseful atmosphere, and its twist ending will keep you on the edge of your seat.
